In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import SDWriter
import pandas as pd
import openpyxl

ChemtasteDB_data = pd.read_excel("data/ChemTastesDB/ChemTastesDB_database.xlsx")
ChemtasteDB_data = ChemtasteDB_data[ChemtasteDB_data["Class taste"].isin(["Bitterness", "Sweetness"])]

#writer = SDWriter("data/ChemTastesDB/ChemTastesDB.sdf")

for _, row in ChemtasteDB_data.iterrows():
    smiles = row['canonical SMILES']
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue  # invalid SMILES

    mol.SetProp("_Name", row['Name'])

    if pd.notna(row['Class taste']):
        mol.SetProp("Class taste", str(row['Class taste']))

    mol = Chem.AddHs(mol)
    embed_result = AllChem.EmbedMolecule(mol, randomSeed=0xf00d)

    if embed_result != 0 or mol.GetNumConformers() == 0:
        print(f"3D embedding failed: {row['Name']}")
        continue

    try:
        AllChem.UFFOptimizeMolecule(mol)
    except Exception as e:
        print(f"UFF optimization failed ({row['Name']}): {e}")
        continue

    writer.write(mol)

writer.close()
print("SDF file saved.")


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdmolfiles import SDWriter

excel_path = "data/ChemTastesDB/ChemTastesDB_database_properties_merge.xlsx"
df = pd.read_excel(excel_path)

#sdf_path = "data/ChemTastesDB/ChemTastesDB_properties.sdf"
writer = SDWriter(sdf_path)

properties = [
    "Class taste",
    "MolecularWeight",
    "XLogP",
    "TPSA",
    "HBondDonorCount",
    "HBondAcceptorCount",
    "RotatableBondCount",
    "HeavyAtomCount",
    "Complexity"
]

invalid_smiles_count = 0
embed_fail_count = 0
success_count = 0


for i, row in df.iterrows():
    smiles = row.get("canonical SMILES")
    if pd.isna(smiles):
        continue
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        invalid_smiles_count += 1
        continue

    mol = Chem.AddHs(mol)
    result = AllChem.EmbedMolecule(mol, randomSeed=0xf00d)
    if result != 0 or mol.GetNumConformers() == 0:
        embed_fail_count += 1
        print(f"3D embedding failed: {row.get('Name', f'Index {i}')}")
        continue

    try:
        AllChem.UFFOptimizeMolecule(mol)
    except Exception as e:
        print(f"UFF optimization failed: {row.get('Name', f'Index {i}')} | {e}")
        continue

    mol.SetProp("_Name", str(row.get("Name", f"Compound_{i}")))
    success_count += 1
    for prop in properties:
        val = row.get(prop)
        if pd.notna(val):
            mol.SetProp(prop, str(val))


    writer.write(mol)

print(f"Invalid SMILES: {invalid_smiles_count}")
print(f"Embedding failures: {embed_fail_count}")
print(f"Molecules written successfully: {success_count}")

writer.close()
print("SDF file saved:", sdf_path)


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import SDWriter
import pandas as pd
import openpyxl

INCLUDED_TASTES = ["Bitterness", "Sweetness", "Tastelessness", "Sourness"]

excel_path = "data/ChemTastesDB/ChemTastesDB_database.xlsx"
df = pd.read_excel(excel_path)

df = df[df["Class taste"].isin(INCLUDED_TASTES)]

#sdf_path = "data/ChemTastesDB/ChemTastesDB_multilabel.sdf"
writer = SDWriter(sdf_path)

embed_fail_count = 0
success_count = 0
invalid_smiles_count = 0

for _, row in df.iterrows():
    smiles = row.get('canonical SMILES')
    if pd.isna(smiles):
        continue

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        invalid_smiles_count += 1
        continue  # invalid SMILES

    name = row.get('Name', '')
    mol.SetProp("_Name", str(name))  # SDF title
    if pd.notna(row.get('Class taste')):
        mol.SetProp("Class taste", str(row.get('Class taste')))

    mol = Chem.AddHs(mol)
    embed_result = AllChem.EmbedMolecule(mol, randomSeed=0xf00d)
    if embed_result != 0 or mol.GetNumConformers() == 0:
        embed_fail_count += 1
        print(f"3D embedding failed: {name}")
        continue

    try:
        AllChem.UFFOptimizeMolecule(mol)
    except Exception as e:
        print(f"UFF optimization failed ({name}): {e}")
        continue

    writer.write(mol)
    success_count += 1

writer.close()
print("SDF file saved:", sdf_path)
print(f"Invalid SMILES: {invalid_smiles_count}")
print(f"Embedding failures: {embed_fail_count}")
print(f"Molecules written successfully: {success_count}")


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdmolfiles import SDWriter

INCLUDED_TASTES = ["Bitterness", "Sweetness", "Tastelessness", "Sourness"]

excel_path = "data/ChemTastesDB/ChemTastesDB_database_properties_merge.xlsx"
df = pd.read_excel(excel_path)

df = df[df["Class taste"].isin(INCLUDED_TASTES)]

#sdf_path = "data/ChemTastesDB/ChemTastesDB_multilabel_properties.sdf"
writer = SDWriter(sdf_path)

properties = [
    "Class taste",
    "MolecularWeight",
    "XLogP",
    "TPSA",
    "HBondDonorCount",
    "HBondAcceptorCount",
    "RotatableBondCount",
    "HeavyAtomCount",
    "Complexity"
]

invalid_smiles_count = 0
embed_fail_count = 0
success_count = 0

for i, row in df.iterrows():
    smiles = row.get("canonical SMILES")
    if pd.isna(smiles):
        continue

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        invalid_smiles_count += 1
        continue

    mol = Chem.AddHs(mol)
    result = AllChem.EmbedMolecule(mol, randomSeed=0xf00d)
    if result != 0 or mol.GetNumConformers() == 0:
        embed_fail_count += 1
        print(f"3D embedding failed: {row.get('Name', f'Index {i}')}")
        continue

    try:
        AllChem.UFFOptimizeMolecule(mol)
    except Exception as e:
        print(f"UFF optimization failed: {row.get('Name', f'Index {i}')} | {e}")
        continue

    mol.SetProp("_Name", str(row.get("Name", f"Compound{i}")))

    for prop in properties:
        val = row.get(prop)
        if pd.notna(val):
            mol.SetProp(prop, str(val))

    writer.write(mol)
    success_count += 1

print(f"Invalid SMILES: {invalid_smiles_count}")
print(f"Embedding failures: {embed_fail_count}")
print(f"Molecules written successfully: {success_count}")

writer.close()
print("SDF file saved:", sdf_path)
